# 19 — Week 5: quantum reservoir computing (QRC) and its controls, validation only

Model code: `src/models/qrc.py`. Only the readout is trained; the reservoir is a fixed random circuit.

* **Per-minute input** (causal, independent of the forecast origin, so each reservoir state is computed **once per
  minute** and cached): u(t) = tanh of the last n one-minute changes divided by s (s = train std of one-minute changes,
  the same scale as the networks' local scaling).
* **Reservoirs** (F = 5n features per minute): `qrc` = fixed random brickwork circuit on n qubits (RY encoding with
  scale a, `depth` layers of random Rot + CZ on alternating neighbour pairs, readout X, Y, Z, ZZ, XX);
  `qrc_noent` = the same without CZ (entanglement ablation); `rf` = size-matched classical random-feature reservoir
  cos(a W u + b); `raw` = no reservoir (the inputs themselves).
* **Aggregation** over the 60 input minutes with exponential kernels lambda in {0, 0.5, 0.8, 0.95}.
* **Readout:** ridge per horizon on standardised features, penalty chosen on validation MAE (kW) from one fixed grid
  for every reservoir type. (Elastic Net, the template's readout, did not converge on the strongly collinear kernel
  aggregates in the synthetic tests; provisional decision.)

Provisional settings: n = 8 qubits, depth 4; input scale a chosen on validation from {pi/32, pi/16, pi/8, pi/4, pi/2, pi} (same grid for
`qrc`, `qrc_noent` and `rf`), then three reservoir seeds (different random circuits / random features). `raw` is
deterministic (one run). Exact statevector simulation, no shots, no noise; **no claim of quantum advantage**.
Selection on validation makes the validation scores slightly optimistic (as for LightGBM). **The test split is not used.**

**Grid extension (run 2, 11 Oct 2026):** run 1 used {pi/8, pi/4, pi/2, pi}; the smallest scale won for `qrc`, `qrc_noent` and `rf` (edge of the grid), so pi/32 and pi/16 were added. All runs are resumable: the 13 runs of run 1 are reused, only the new scales (and the seeds of a new best scale) are computed. Run-1 outputs are kept in `19_week5_qrc_outputs_run1.md`. Provisional rule: no further extension; if pi/32 still wins, that is recorded as a finding (the readout prefers the near-linear regime of the reservoir).

In [11]:
# Cell 1: setup
import sys, json, time, math, shutil, warnings, traceback
from pathlib import Path
import numpy as np
import polars as pl
import torch

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import pennylane as qml
from models import rnn as R
from models import qrc as QR
from evaluation import compare as C

GROUP = "week5_qrc"
REP = ROOT / "reports" / "week5"
REP.mkdir(parents=True, exist_ok=True)
N_QUBITS, DEPTH = 8, 4
SCALES = (math.pi / 32, math.pi / 16, math.pi / 8, math.pi / 4, math.pi / 2, math.pi)   # run 2: pi/32, pi/16 added
SEEDS = [0, 1, 2]
KINDS = ["qrc", "qrc_noent", "rf", "raw"]
DATA_CFG = {"scaling": "local", "calendar": False, "covariates": False}
DATASET = "data/processed/model_dataset.parquet"
SUMMARY = {"n_qubits": N_QUBITS, "depth": DEPTH, "scales": SCALES, "lambdas": QR.LAMBDAS,
           "ridge_grid": QR.RIDGE_GRID}
print("pennylane", qml.__version__, "| features per minute (qrc):", QR.n_features("qrc", N_QUBITS),
      "| window features:", QR.n_features("qrc", N_QUBITS) * len(QR.LAMBDAS))

pennylane 0.45.1 | features per minute (qrc): 40 | window features: 160


In [12]:
# Cell 2: unit checks (no real data)
checks = {}
rng = np.random.default_rng(0)
n = 4
U = np.tanh(rng.normal(size=(5, n)))
F = QR.quantum_features(U, depth=3, scale=1.1, seed=1)
ang = QR.circuit_angles(n, 3, 1)
dev = qml.device("default.qubit", wires=n)
@qml.qnode(dev)
def single(u):
    for i in range(n):
        qml.RY(1.1 * u[i], wires=i)
    for d in range(3):
        for i in range(n):
            qml.Rot(*ang[d, i], wires=i)
        for i in range(d % 2, n - 1, 2):
            qml.CZ(wires=[i, i + 1])
    return [qml.expval(qml.PauliZ(i)) for i in range(n)] + [qml.expval(qml.PauliX(0) @ qml.PauliX(1))]
ref = np.array([single(u) for u in U])
d1 = float(np.abs(ref[:, :n] - F[:, 2 * n:3 * n]).max())
d2 = float(np.abs(ref[:, n] - F[:, 3 * n + len(QR.pairs(n))]).max())
checks["broadcast_equals_single_circuit"] = d1 < 1e-6 and d2 < 1e-6
print(f"(a) broadcast reservoir vs one-sample circuit: max |diff| Z {d1:.1e}, XX {d2:.1e}")

Fa = QR.quantum_features(U, depth=3, seed=1, entangle=False)
U2 = U.copy(); U2[:, 0] += 0.5
Fb = QR.quantum_features(U2, depth=3, seed=1, entangle=False)
single_q = [j for j in range(3 * n) if j % n != 0]
d3 = float(np.abs(Fa[:, single_q] - Fb[:, single_q]).max())
checks["noent_is_product_state"] = d3 < 1e-6
print(f"(b) no entanglement: change of single-qubit features of qubits 1..3 when only u_0 changes: {d3:.1e}")

X = rng.normal(size=(400, 12)); X[:, 1] = X[:, 0] + 1e-3 * rng.normal(size=400)
y = X @ rng.normal(size=12) + rng.normal(size=400)
from sklearn.linear_model import Ridge
fit = QR.fit_readout(X, y, X[:50], y[:50], np.zeros(50), 1.0, ridge_grid=(0.01,))
Z = (X - X.mean(0)) / X.std(0)
sk = Ridge(alpha=0.01 * len(y)).fit(Z, y)
d4 = float(np.abs(sk.coef_ - fit["coef"]).max())
checks["ridge_closed_form_equals_sklearn"] = d4 < 1e-6
print(f"(c) closed-form ridge vs sklearn Ridge: max |coef diff| {d4:.1e}")
for k in KINDS:
    print(f"(d) {k:10s} features per minute {QR.n_features(k, N_QUBITS):>3} | window features "
          f"{QR.n_features(k, N_QUBITS) * len(QR.LAMBDAS):>4} | readout params (3 horizons) "
          f"{3 * (QR.n_features(k, N_QUBITS) * len(QR.LAMBDAS) + 1)}")
UNIT_OK = all(checks.values())
print(checks, "\nUNIT CHECKS PASSED" if UNIT_OK else "\nUNIT CHECKS FAILED: send this output")

(a) broadcast reservoir vs one-sample circuit: max |diff| Z 1.6e-08, XX 3.6e-09
(b) no entanglement: change of single-qubit features of qubits 1..3 when only u_0 changes: 0.0e+00
(c) closed-form ridge vs sklearn Ridge: max |coef diff| 4.0e-14
(d) qrc        features per minute  40 | window features  160 | readout params (3 horizons) 483
(d) qrc_noent  features per minute  40 | window features  160 | readout params (3 horizons) 483
(d) rf         features per minute  40 | window features  160 | readout params (3 horizons) 483
(d) raw        features per minute   8 | window features   32 | readout params (3 horizons) 99
{'broadcast_equals_single_circuit': True, 'noent_is_product_state': True, 'ridge_closed_form_equals_sklearn': True} 
UNIT CHECKS PASSED


In [13]:
# Cell 3: DRY RUN on synthetic data (4 qubits, every reservoir type; deleted afterwards)

def synth(n=30000, seed=0):
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    x = np.zeros(n)
    e = rng.normal(0, 0.4, n)
    for i in range(1, n):
        x[i] = 0.9 * x[i - 1] + e[i]
    P = 30 + np.cumsum(rng.normal(0, 0.01, n)) + x - 0.3 * (t % 5 == 0)
    valid = np.ones(n, bool)
    for g0 in rng.choice(np.arange(500, n - 500), 6, replace=False):
        valid[g0:g0 + 150] = False
    i_tr, i_va = int(0.7 * n), int(0.85 * n)
    split = np.where(t < i_tr, "train", np.where(t < i_va, "val", "test"))
    c = np.concatenate([[0], np.cumsum(~valid)])
    win_ok = np.zeros(n, bool)
    win_ok[R.INPUT_LEN - 1:] = (c[R.INPUT_LEN:] - c[:n - R.INPUT_LEN + 1]) == 0
    Pv = np.where(valid, P, np.nan)
    L24 = pl.Series(Pv).fill_nan(None).rolling_mean(1440, min_samples=720).to_numpy()
    cols = {"minute_epoch": t * 60, "P_kW": Pv, "valid": valid, "split": split, "mod5": (t % 5).astype(np.int64),
            "hour_sin": np.sin(2 * np.pi * (t // 60 % 24) / 24), "hour_cos": np.cos(2 * np.pi * (t // 60 % 24) / 24),
            "L24_kW": L24}
    thr = {}
    for h in R.HORIZONS:
        ok = win_ok.copy()
        ok[n - h:] = False
        ok[:n - h] &= valid[h:]
        sp = np.full(n, None, dtype=object)
        tt = np.arange(n - h)
        sp[:n - h] = np.where(tt + h < i_tr, "train", np.where((tt >= i_tr) & (tt + h < i_va), "val",
                              np.where(tt >= i_va, "test", None)))
        ok &= sp != None  # noqa: E711
        rel = np.full(n, np.nan)
        rel[:n - h] = np.abs(P[h:] - P[:n - h]) / L24[:n - h]
        trm = ok & (sp == "train") & np.isfinite(rel)
        thr[str(h)] = {"rel_p95": float(np.quantile(rel[trm], 0.95))}
        ramp = np.where(ok & np.isfinite(rel), (rel > thr[str(h)]["rel_p95"]).astype(float), np.nan)
        cols[f"ok_h{h}"], cols[f"split_h{h}"], cols[f"ramp_h{h}"] = ok, sp.tolist(), ramp
    df = pl.DataFrame(cols).with_columns(pl.col("P_kW").fill_nan(None), pl.col("L24_kW").fill_nan(None))
    return df, {"thresholds": thr}


assert UNIT_OK, "run cell 2 first"
DRY = "_dryrun_week5_qrc"
sm, smeta = synth()
sdata = R.Data(sm, DATA_CFG)
sU = QR.minute_inputs(sdata, sm, 4)
dry_ok = True
try:
    for k in KINDS:
        res = QR.run_and_score(ROOT, sm, smeta, sdata, sU, k, math.pi / 2, 0, DRY, depth=DEPTH,
                               tracking_mode="disabled", log=lambda s: None)
        out = ROOT / "experiments" / DRY / (f"{QR.variant_name(k, 4, DEPTH)}_{QR.scale_tag(math.pi / 2)}_s0"
                                            if k != "raw" else f"{QR.variant_name(k, 4, DEPTH)}_s0")
        files = ["config.yaml", "predictions_val.parquet", "readout.npz", "metrics.json"]
        missing = [f for f in files if not (out / f).exists()]
        finite = all(np.isfinite(r_["MAE"]) for r_ in res["val"])
        dry_ok &= (not missing) and finite
        print(f"{QR.variant_name(k, 4, DEPTH):16s} files ok {not missing} | finite {finite} | skill " +
              " / ".join(f"{r_['skill_MAE']:+.3f}" for r_ in res["val"]))
except Exception:
    dry_ok = False
    traceback.print_exc()
finally:
    shutil.rmtree(ROOT / "experiments" / DRY, ignore_errors=True)
    shutil.rmtree(ROOT / "experiments" / "runs" / DRY, ignore_errors=True)
    log_p = REP / "quantum_variants_log.csv"
    if log_p.exists():
        lg = pl.read_csv(log_p, infer_schema_length=0)
        lg = lg.filter(pl.col("group") != DRY) if "group" in lg.columns else lg
        lg.write_csv(log_p)
print("DRY RUN PASSED" if dry_ok else "DRY RUN FAILED: send this output")

qrc_q4_d4        files ok True | finite True | skill +0.029 / +0.047 / +0.079
qrc_noent_q4_d4  files ok True | finite True | skill +0.034 / +0.048 / +0.079
ctrl_rf_q4_f20   files ok True | finite True | skill +0.035 / +0.063 / +0.112
ctrl_raw_q4      files ok True | finite True | skill +0.034 / +0.059 / +0.097
DRY RUN PASSED


In [14]:
# Cell 4: real data, inputs and reservoir timing (10,000 minutes per configuration)
assert dry_ok, "run cell 3 first"
md = pl.read_parquet(ROOT / DATASET)
meta = json.loads((ROOT / "data/processed/model_dataset_meta.json").read_text())
data = R.Data(md, DATA_CFG)
U = QR.minute_inputs(data, md, N_QUBITS)
print(f"minutes {len(U):,} | inputs per minute {U.shape[1]} | share of exact zeros {float((U == 0).mean()):.3f} | "
      f"|u| quantiles 50/95/99%: {np.round(np.quantile(np.abs(U[U != 0]), [0.5, 0.95, 0.99]), 3)}")
tim = []
for nq in (2, 4, 6, 8):
    for ent in (True, False):
        Ui = QR.minute_inputs(data, md, nq)[:10000]
        t0 = time.time()
        QR.quantum_features(Ui, depth=DEPTH, seed=0, entangle=ent)
        dt = time.time() - t0
        tim.append({"n_qubits": nq, "entangle": ent, "s_per_10k": round(dt, 2),
                    "est_min_all_minutes": round(dt * len(U) / 10000 / 60, 2)})
tim = pl.DataFrame(tim)
print(tim)
tim.write_csv(REP / "qrc_timing.csv")
SUMMARY["timing"] = tim.to_dicts()

minutes 303,840 | inputs per minute 8 | share of exact zeros 0.070 | |u| quantiles 50/95/99%: [0.359 0.957 0.998]
shape: (8, 4)
┌──────────┬──────────┬───────────┬─────────────────────┐
│ n_qubits ┆ entangle ┆ s_per_10k ┆ est_min_all_minutes │
│ ---      ┆ ---      ┆ ---       ┆ ---                 │
│ i64      ┆ bool     ┆ f64       ┆ f64                 │
╞══════════╪══════════╪═══════════╪═════════════════════╡
│ 2        ┆ true     ┆ 0.02      ┆ 0.01                │
│ 2        ┆ false    ┆ 0.02      ┆ 0.01                │
│ 4        ┆ true     ┆ 0.11      ┆ 0.06                │
│ 4        ┆ false    ┆ 0.1       ┆ 0.05                │
│ 6        ┆ true     ┆ 0.64      ┆ 0.32                │
│ 6        ┆ false    ┆ 0.55      ┆ 0.28                │
│ 8        ┆ true     ┆ 3.08      ┆ 1.56                │
│ 8        ┆ false    ┆ 2.54      ┆ 1.29                │
└──────────┴──────────┴───────────┴─────────────────────┘


In [15]:
# Cell 5: SCALE SCREEN, seed 0 (input scale a for qrc, qrc_noent and rf; raw has none). Resumable.
# 19 runs (12 with an 8-qubit circuit, about 2 min each; the 13 runs of run 1 are skipped). A choice at the grid edge is flagged.
T = time.time()
scr = []
for k in KINDS:
    for a in (SCALES if k != "raw" else (math.pi / 2,)):
        res = QR.run_and_score(ROOT, md, meta, data, U, k, a, 0, GROUP, depth=DEPTH, screen=True)
        scr.append({"kind": k, "variant": QR.variant_name(k, N_QUBITS, DEPTH), "scale": a,
                    **{f"skill_h{r_['h']}": r_["skill_MAE"] for r_ in res["val"]},
                    "val_MAE_mean": float(np.mean([r_["MAE"] for r_ in res["val"]])),
                    "ridge_at_edge": res["info"].get("ridge_at_grid_edge")})
scr = pl.DataFrame(scr).sort("kind", "val_MAE_mean")
print(scr.with_columns(pl.col("scale").round(4), pl.col("val_MAE_mean").round(5),
                       *[pl.col(f"skill_h{h}").round(4) for h in R.HORIZONS]))
scr.write_csv(REP / "qrc_scale_screen_s0.csv")
BEST_SCALE = {k: float(scr.filter(pl.col("kind") == k).sort("val_MAE_mean")["scale"][0]) for k in KINDS}
edge = [k for k in ("qrc", "qrc_noent", "rf") if BEST_SCALE[k] in (min(SCALES), max(SCALES))]
print("chosen scale at the edge of the grid:", edge or "none",
      "| rule: no further extension; a choice at the lower edge is reported as near-linear regime" if edge else "")
print("\nchosen input scale per reservoir type (provisional, validation):",
      {k: round(v, 4) for k, v in BEST_SCALE.items()}, f"| {(time.time() - T) / 60:.1f} min")
SUMMARY["scale_screen"] = scr.to_dicts()
SUMMARY["best_scale"] = BEST_SCALE

[15:53:17] qrc_q8_d4_a0p098_s0: reservoir + readout
[15:54:55] qrc_q8_d4_a0p098_s0: features 160 | readout params 483 | reservoir 88 s | val MAE 0.2650 / 0.4122 / 0.5859 kW | skill +0.075 / +0.041 / +0.031
[15:54:55] qrc_q8_d4_a0p196_s0: reservoir + readout
[15:56:32] qrc_q8_d4_a0p196_s0: features 160 | readout params 483 | reservoir 88 s | val MAE 0.2653 / 0.4123 / 0.5858 kW | skill +0.074 / +0.041 / +0.031
[15:56:32] qrc_q8_d4_a0p393_s0: done earlier, skipped
[15:56:32] qrc_q8_d4_a0p785_s0: done earlier, skipped
[15:56:32] qrc_q8_d4_a1p571_s0: done earlier, skipped
[15:56:32] qrc_q8_d4_a3p142_s0: done earlier, skipped
[15:56:32] qrc_noent_q8_d4_a0p098_s0: reservoir + readout
[15:57:59] qrc_noent_q8_d4_a0p098_s0: features 160 | readout params 483 | reservoir 77 s | val MAE 0.2652 / 0.4126 / 0.5872 kW | skill +0.075 / +0.040 / +0.029
[15:57:59] qrc_noent_q8_d4_a0p196_s0: reservoir + readout
[15:59:30] qrc_noent_q8_d4_a0p196_s0: features 160 | readout params 483 | reservoir 77 s | val M

In [16]:
# Cell 6: THREE RESERVOIR SEEDS at the chosen scale (seed 0 reused from the screen). Resumable.
T = time.time()
for seed in SEEDS:
    for k in KINDS:
        if k == "raw" and seed > 0:
            continue                                   # deterministic: no random reservoir
        try:
            QR.run_and_score(ROOT, md, meta, data, U, k, BEST_SCALE[k], seed, GROUP, depth=DEPTH)
        except Exception:
            print(f"!!! {k} seed {seed} failed:")
            traceback.print_exc(limit=5)
print(f"\nall runs handled in {(time.time() - T) / 60:.1f} min")


def run_dir(k, seed):
    base = QR.variant_name(k, N_QUBITS, DEPTH)
    return ROOT / "experiments" / GROUP / (f"{base}_{QR.scale_tag(BEST_SCALE[k])}_s{seed}" if k != "raw" else f"{base}_s0")

[15:59:58] qrc_q8_d4_a0p098_s0: done earlier, skipped
[15:59:58] qrc_noent_q8_d4_a0p196_s0: done earlier, skipped
[15:59:58] ctrl_rf_q8_f40_a0p098_s0: done earlier, skipped
[15:59:58] ctrl_raw_q8_s0: done earlier, skipped
[15:59:58] qrc_q8_d4_a0p098_s1: reservoir + readout
[16:01:37] qrc_q8_d4_a0p098_s1: features 160 | readout params 483 | reservoir 87 s | val MAE 0.2650 / 0.4121 / 0.5861 kW | skill +0.075 / +0.041 / +0.031
[16:01:37] qrc_noent_q8_d4_a0p196_s1: reservoir + readout
[16:03:07] qrc_noent_q8_d4_a0p196_s1: features 160 | readout params 483 | reservoir 75 s | val MAE 0.2648 / 0.4122 / 0.5874 kW | skill +0.076 / +0.041 / +0.029
[16:03:07] ctrl_rf_q8_f40_a0p098_s1: reservoir + readout
[16:03:19] ctrl_rf_q8_f40_a0p098_s1: features 160 | readout params 483 | reservoir 0 s | val MAE 0.2644 / 0.4116 / 0.5863 kW | skill +0.077 / +0.043 / +0.031
[16:03:19] qrc_q8_d4_a0p098_s2: reservoir + readout
[16:04:58] qrc_q8_d4_a0p098_s2: features 160 | readout params 483 | reservoir 88 s | va

In [17]:
# Cell 7: table (mean +- sd over reservoir seeds), merged with the Week 4 benchmark and the hybrid QNN (if available)
rows = []
for k in KINDS:
    for seed in (SEEDS if k != "raw" else [0]):
        p = run_dir(k, seed) / "metrics.json"
        if not p.exists():
            print("missing:", p.parent.name)
            continue
        r = json.loads(p.read_text())
        for v in r["val"]:
            rows.append({"model": QR.variant_name(k, N_QUBITS, DEPTH), "kind": k, "seed": seed,
                         **{c: v.get(c) for c in ["h", "MAE", "RMSE", "sMAPE", "R2_delta", "skill_MAE", "MAE_ramp",
                                                  "MAE_normal", "ramp_F1", "MAE_no_mod0_target", "DM_p"]},
                         "n_params": r["info"]["n_params"], "reservoir_s": r["info"]["reservoir_s"],
                         "readout_s": r["info"]["readout_s"], "ridge": r["info"][f"ridge_h{v['h']}"]})
runs = pl.DataFrame(rows)
runs.write_csv(REP / "qrc_val_runs.csv")
summ = (runs.group_by("model", "kind", "h").agg(
            pl.len().alias("n_seeds"), pl.col("MAE").mean().alias("MAE_mean"), pl.col("MAE").std().alias("MAE_sd"),
            *[pl.col(c).mean() for c in ["RMSE", "sMAPE", "R2_delta", "skill_MAE", "MAE_ramp", "MAE_normal", "ramp_F1",
                                         "MAE_no_mod0_target"]],
            pl.col("DM_p").max(), pl.col("n_params").first(), pl.col("reservoir_s").mean(), pl.col("readout_s").mean())
          .with_columns(pl.when(pl.col("kind").str.starts_with("qrc")).then(pl.lit("quantum reservoir"))
                          .otherwise(pl.lit("classical control")).alias("family"))
          .sort("h", "MAE_mean"))
summ.write_csv(REP / "qrc_val_summary.csv")
pl.Config.set_tbl_rows(80); pl.Config.set_tbl_cols(20); pl.Config.set_tbl_width_chars(250)
print(summ.select("h", "model", "n_seeds", pl.col("MAE_mean").round(4), pl.col("MAE_sd").round(4),
                  pl.col("skill_MAE").round(3), pl.col("MAE_ramp").round(3), pl.col("MAE_normal").round(4),
                  pl.col("ramp_F1").round(3), "n_params", pl.col("reservoir_s").round(0)))
w4 = pl.read_csv(ROOT / "reports" / "week4" / "benchmark_val_week4_all.csv")
w5q = REP / "benchmark_val_week5_qnn.csv"
base = pl.read_csv(w5q) if w5q.exists() else w4
new = summ.with_columns(pl.lit(False).alias("covariates")).select([c for c in w4.columns])
bench = pl.concat([base.select(w4.columns), new.with_columns([pl.col(c).cast(w4.schema[c]) for c in w4.columns])])
bench = bench.sort("h", "MAE_mean").with_columns(pl.col("MAE_mean").rank("ordinal").over("h").alias("rank"))
show = ["lightgbm", "patchtst", "gru_uni", "ar_lags", "persistence"] + summ["model"].unique().to_list()
show += [m for m in bench["model"].unique().to_list() if m.startswith(("hqnn", "ctrl_mlp", "ctrl_bottleneck"))]
for h in R.HORIZONS:
    print(f"\n=== h = {h} min: validation ranking ({bench.filter(pl.col('h') == h).height} models) ===")
    print(bench.filter((pl.col("h") == h) & pl.col("model").is_in(show))
               .select("rank", "family", "model", "n_seeds", pl.col("MAE_mean").round(4), pl.col("skill_MAE").round(3)))
SUMMARY["val_summary"] = summ.to_dicts()

shape: (12, 11)
┌─────┬─────────────────┬─────────┬──────────┬────────┬───────────┬──────────┬────────────┬─────────┬──────────┬─────────────┐
│ h   ┆ model           ┆ n_seeds ┆ MAE_mean ┆ MAE_sd ┆ skill_MAE ┆ MAE_ramp ┆ MAE_normal ┆ ramp_F1 ┆ n_params ┆ reservoir_s │
│ --- ┆ ---             ┆ ---     ┆ ---      ┆ ---    ┆ ---       ┆ ---      ┆ ---        ┆ ---     ┆ ---      ┆ ---         │
│ i64 ┆ str             ┆ u32     ┆ f64      ┆ f64    ┆ f64       ┆ f64      ┆ f64        ┆ f64     ┆ i64      ┆ f64         │
╞═════╪═════════════════╪═════════╪══════════╪════════╪═══════════╪══════════╪════════════╪═════════╪══════════╪═════════════╡
│ 1   ┆ ctrl_rf_q8_f40  ┆ 3       ┆ 0.2645   ┆ 0.0005 ┆ 0.077     ┆ 1.415    ┆ 0.2121     ┆ 0.048   ┆ 483      ┆ 0.0         │
│ 1   ┆ qrc_noent_q8_d4 ┆ 3       ┆ 0.2649   ┆ 0.0001 ┆ 0.076     ┆ 1.434    ┆ 0.2116     ┆ 0.031   ┆ 483      ┆ 76.0        │
│ 1   ┆ qrc_q8_d4       ┆ 3       ┆ 0.2652   ┆ 0.0004 ┆ 0.075     ┆ 1.42     ┆ 0.2125     ┆ 0.0

In [18]:
# Cell 8: Diebold-Mariano (Holm per horizon) and ramp breakdown, seed 0; reference: ridge AR on the 60 lags
pred = C.load_predictions(ROOT, seed=0, log=lambda s: None)
names = [QR.variant_name(k, N_QUBITS, DEPTH) for k in KINDS]
parts = [pred] + [C._std(pl.read_parquet(run_dir(k, 0) / "predictions_val.parquet"), nm) for k, nm in zip(KINDS, names)]
pred = pl.concat(parts)
REFS = [m for m in ["ar_lags", "patchtst", "persistence"] if m in pred["model"].unique().to_list()]
print("references found:", REFS)
MODELS = names + REFS
al, cov = C.align(pred.filter(pl.col("model").is_in(MODELS)))
dm = C.dm_pairs(al, {h: MODELS for h in R.HORIZONS})
dm.write_csv(REP / "qrc_dm_pairs_val.csv")
print(dm.filter(pl.col("model_a").str.starts_with("qrc") | pl.col("model_b").str.starts_with("qrc"))
        .select("h", "model_a", "model_b", pl.col("MAE_a").round(4), pl.col("MAE_b").round(4), pl.col("DM_stat").round(2),
                "DM_p_holm", "better", "significant_5pct"))
rb = C.ramp_breakdown(C.add_labels(al, md), meta["thresholds"])
rb.write_csv(REP / "qrc_ramp_breakdown_val.csv")
print(rb.select("h", "model", "n_ramp", pl.col("MAE_normal").round(4), pl.col("MAE_ramp").round(3),
                pl.col("skill_normal").round(3), pl.col("skill_ramp").round(3), pl.col("ramp_F1").round(3)))
SUMMARY["dm"] = dm.to_dicts()

references found: ['ar_lags', 'patchtst', 'persistence']
shape: (33, 9)
┌─────┬─────────────────┬─────────────────┬────────┬────────┬─────────┬────────────┬─────────────────┬──────────────────┐
│ h   ┆ model_a         ┆ model_b         ┆ MAE_a  ┆ MAE_b  ┆ DM_stat ┆ DM_p_holm  ┆ better          ┆ significant_5pct │
│ --- ┆ ---             ┆ ---             ┆ ---    ┆ ---    ┆ ---     ┆ ---        ┆ ---             ┆ ---              │
│ i64 ┆ str             ┆ str             ┆ f64    ┆ f64    ┆ f64     ┆ f64        ┆ str             ┆ bool             │
╞═════╪═════════════════╪═════════════════╪════════╪════════╪═════════╪════════════╪═════════════════╪══════════════════╡
│ 1   ┆ qrc_q8_d4       ┆ qrc_noent_q8_d4 ┆ 0.265  ┆ 0.2649 ┆ 0.41    ┆ 0.680975   ┆ qrc_noent_q8_d4 ┆ false            │
│ 1   ┆ qrc_q8_d4       ┆ ctrl_rf_q8_f40  ┆ 0.265  ┆ 0.2642 ┆ 3.54    ┆ 0.002835   ┆ ctrl_rf_q8_f40  ┆ true             │
│ 1   ┆ qrc_q8_d4       ┆ ctrl_raw_q8     ┆ 0.265  ┆ 0.27   ┆ -7.99   ┆ 1.

In [19]:
# Cell 9: summary
(REP / "qrc_val.json").write_text(json.dumps(SUMMARY, indent=1, default=str))
print("written:", REP / "qrc_val.json")

written: /Users/aryaneuse/Desktop/Uni Project/datacenter-power-forecasting/reports/week5/qrc_val.json


In [21]:
# Cell 10: export outputs (save the notebook first with Cmd+S, then run this cell)
%run ../src/tools/nb_outputs_to_md.py 19_week5_qrc.ipynb

written 19_week5_qrc_outputs.md | code cells 11 | figures 0
